In [1]:
#caricamento del dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, pearsonr
import numpy as np


df = pd.read_csv('../../../data/processed/DatasetFinali/heloc_dataset_cleanedDLLM.csv')


In [2]:
#Cerchiamo le colonne "miste" fatte sia da valori categorici che valori numerici

#lista per salvare le colonne
mixed_columns = []

#scansioniamo le colonne del dataset
for col in df.columns:
    #facciamo una prova e convertiamo in numerico le colonne
    numeric_conversion = pd.to_numeric(df[col], errors = 'coerce')

    #contiamo i risultati
    numeric_count = numeric_conversion.notna().sum()
    non_numeric_count = numeric_conversion.isna().sum()

    #prendiamo le colonne che hanno sia valori numerici che non numerici e quindi miste
    if numeric_count > 0 and non_numeric_count > 0:
        mixed_columns.append(col)

#stampa del risultato
print(f"Colonne MISTE trovate: {len(mixed_columns)}\n")
print("Nomi delle colonne miste:")
for col in mixed_columns:
    print(f" {col}")


Colonne MISTE trovate: 9

Nomi delle colonne miste:
 MSinceOldestTradeOpen
 MSinceMostRecentDelq
 MSinceMostRecentInqexcl7days
 NetFractionRevolvingBurden
 NetFractionInstallBurden
 NumRevolvingTradesWBalance
 NumInstallTradesWBalance
 NumBank2NatlTradesWHighUtilization
 PercentTradesWBalance


In [3]:
#separiamo che variabili categoriche e numeriche dalle celle miste calcolate nella cella 2
numeric_components = {}
categorical_components = {}


for col in mixed_columns:
    #convertiamo a numerico 
    numeric_conversion = pd.to_numeric(df[col], errors = 'coerce')

    #separiamo i due componenti da quelli che so usciti na dalla conversione
    numeric_components[col] = numeric_conversion.dropna() #solo numeriche 
    categorical_components[col] = df[numeric_conversion.isna()][col] #solo categoriche



In [4]:
#Analisi delle relazioni tra feature

#togliamo la variabile target
df_features = df.drop(columns=['RiskPerformance'])

#estendi i dizionari a tutte le colonne del dataset
for col in df_features.columns:
    if col not in mixed_columns:
        numeric_components[col] = pd.to_numeric(df_features[col], errors = 'coerce')
        categorical_components[col] = None

#lista di tutte le colonne numeriche
all_cols = list(df_features.columns)


#CORRELAZIONE PEARSON
#Correlazione di Pearson fatta tramite tutte le feature "normali" e con le osservazioni numeriche delle colonne miste
pearson_data = []
#costruzione della matrice 
for col1 in all_cols:
    pearson_row = {}
    for col2 in all_cols:
        vals1 = numeric_components.get(col1)
        vals2 = numeric_components.get(col2)
        #questo serve per prendere le colonne che sulla stessa riga hanno due valori numerici
        #questo è obbligatorio per via del fatto che anche se filtrate hanno valori diversi in indici diversi
        common_idx = vals1.index.intersection(vals2.index)
        if len(common_idx) > 1:
            r, p = pearsonr(vals1.loc[common_idx], vals2.loc[common_idx])
            pearson_row[col2] = r
        else:
            pearson_row[col2] = np.nan
    pearson_data.append(pearson_row)

df_pearson = pd.DataFrame(pearson_data, index=all_cols)




#CALCOLO DI CRAMER
cramer_data = []
#calcoliamo cramer solo se entrambe le colonne sono miste dato che sono le uniche con valori categorici nel nostro dataset (oltre la target)
for col1 in all_cols:
    cramer_row = {}
    for col2 in all_cols:
        if col1 in mixed_columns and col2 in mixed_columns:
            data1 = categorical_components[col1]
            data2 = categorical_components[col2]
            #trovare gli indici comuni per allineare i dati
            common_idx = data1.index.intersection(data2.index)
            if len(common_idx) > 0:
                #filtra ai dati comuni
                data1_common = data1.loc[common_idx]
                data2_common = data2.loc[common_idx]
                #tabella di contingenza per le due variabili categoriche
                contingency = pd.crosstab(data1_common, data2_common)
                #test del chi quadrato per la tabella di contingenza
                chi2, p, dof, expected = chi2_contingency(contingency)
                #numero di osservazioni
                n = contingency.sum().sum()
                min_dim = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
                #formula di cramer
                cramer_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
                cramer_row[col2] = cramer_v
            else:
                cramer_row[col2] = np.nan
        else:
            cramer_row[col2] = np.nan
    cramer_data.append(cramer_row)
df_cramer = pd.DataFrame(cramer_data, index=all_cols)


#CALCOLO DI ETA

eta_data = []
#per ogni colonna fattore (variabile categorica indipendente) calcola eta con ogni colonna come risposta (variabile numerica dipendente)
for factor_col in all_cols:
    eta_row = {}
    for response_col in all_cols:
        #la correlazione di una variabile con se stessa è sempre 1
        if factor_col == response_col:
            eta_row[response_col] = 1.0
        #eta ha senso solo con valori categorici quindi prendiamo i valori categorici delle colonne miste visto che nelle altre feature abbiamo solo valori numerici
        elif factor_col in mixed_columns:
            #fattore: componente categorica della mista
            #risposta: componente numerica di response_col
            factor = categorical_components[factor_col]
            response = numeric_components[response_col]
            #usiamo sempre un indici comune per confrontare le due serie di dati
            common_idx = factor.index.intersection(response.index)
            #allinea i dati per il calcolo di eta
            if len(common_idx) > 0:
                factor_subset = factor.loc[common_idx]
                response_subset = response.loc[common_idx]
                #media generale delle variabili numeriche
                grand_mean = response_subset.mean()
                #somma dei quadrati totale  rappresenta la variabilità totale della risposta
                ss_total = np.sum((response_subset - grand_mean) ** 2)
                #somma dei quadrati tra i gruppi, spiega la variabilità del fattore categorico
                ss_between = sum(len(response_subset[factor_subset == g]) * (response_subset[factor_subset == g].mean() - grand_mean) ** 2 for g in factor_subset.unique())
                #formula dell'eta
                eta = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0
                eta_row[response_col] = eta
            else:
                eta_row[response_col] = np.nan
        else:
            eta_row[response_col] = np.nan
    eta_data.append(eta_row)
df_eta = pd.DataFrame(eta_data, index=all_cols)


# SALVATAGGIO CSV E CREAZIONE HEATMAP
import os

output_dir = "../../../data/processed/Fase1/AnalysisDLLM"
os.makedirs(output_dir, exist_ok=True)

dataset_name = "heloc_DLLM"

# Salvataggio CSV
print("\n" + "="*70)
print("SALVATAGGIO MATRICI E VISUALIZZAZIONE")
print("="*70)
print("\nSalvataggio CSV...")

df_pearson.to_csv(os.path.join(output_dir, f"{dataset_name}_pearson_matrix.csv"))
print(f"  ✓ {dataset_name}_pearson_matrix.csv")

df_cramer.to_csv(os.path.join(output_dir, f"{dataset_name}_cramersv_matrix.csv"))
print(f"  ✓ {dataset_name}_cramersv_matrix.csv")

df_eta.to_csv(os.path.join(output_dir, f"{dataset_name}_eta_matrix.csv"))
print(f"  ✓ {dataset_name}_eta_matrix.csv")

# Creazione heatmap PNG
print("\nCreazione e salvataggio heatmap...")

# Pearson heatmap
plt.figure(figsize=(18, 16))
sns.heatmap(df_pearson, annot=True, cmap='coolwarm', vmin=-1, vmax=1, square=True, cbar_kws={'label': 'Pearson r'}, annot_kws={'size': 7})
plt.title("Pearson Correlation Matrix (Componenti Numerici)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{dataset_name}_pearson_heatmap.png"), dpi=300)
plt.close()
print(f"  ✓ {dataset_name}_pearson_heatmap.png")

# Cramér's V heatmap
plt.figure(figsize=(18, 16))
sns.heatmap(df_cramer, annot=True, cmap='viridis', vmin=0, vmax=1, square=True, cbar_kws={'label': "Cramér's V"})
plt.title("Cramér's V Matrix (Componenti Categorici)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{dataset_name}_cramersv_heatmap.png"), dpi=300)
plt.close()
print(f"  ✓ {dataset_name}_cramersv_heatmap.png")

# Eta heatmap
plt.figure(figsize=(18, 16))
sns.heatmap(df_eta, annot=True, cmap='RdYlGn', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Eta Coefficient'})
plt.title("Eta Coefficient Matrix (Misura Complessiva)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{dataset_name}_eta_heatmap.png"), dpi=300)
plt.close()
print(f"  ✓ {dataset_name}_eta_heatmap.png")

print("\n" + "="*70)
print("✓ ANALISI FEATURE-TO-FEATURE COMPLETATA!")
print("="*70)
print(f"Output salvato in: {output_dir}\n")
        
            



SALVATAGGIO MATRICI E VISUALIZZAZIONE

Salvataggio CSV...
  ✓ heloc_DLLM_pearson_matrix.csv
  ✓ heloc_DLLM_cramersv_matrix.csv
  ✓ heloc_DLLM_eta_matrix.csv

Creazione e salvataggio heatmap...
  ✓ heloc_DLLM_pearson_heatmap.png
  ✓ heloc_DLLM_cramersv_heatmap.png
  ✓ heloc_DLLM_eta_heatmap.png

✓ ANALISI FEATURE-TO-FEATURE COMPLETATA!
Output salvato in: ../../../data/processed/Fase1/AnalysisDLLM



In [5]:
# Tabella Correlazioni Pearson Tutte le coppie (Calcolata solo sulla componente numerica delle colonne miste)
import pandas as pd
import numpy as np

print("\n" + "="*70)
print("TABELLA CORRELAZIONI PEARSON ((Tutte le coppie - Calcolata solo sulla componente numerica delle colonne miste)")
print("="*70)

# Caricamento del dataset per il conteggio delle osservazioni valide (n_valid)
df_data = pd.read_csv('../../../data/processed/DatasetFinali/heloc_dataset_cleanedDLLM.csv')

corr = df_pearson.copy()

# Filtro e selezione delle coppie fortemente correlate
pairs = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        r = corr.iloc[i, j]
        if pd.notna(r):
            feat1 = corr.columns[i]
            feat2 = corr.columns[j]
            
            # Calcolo n_valid: numero di osservazioni dove
            # entrambe le feature hanno valori numerici validi
            col1 = pd.to_numeric(df_data[feat1], errors='coerce')
            col2 = pd.to_numeric(df_data[feat2], errors='coerce')
            n_valid = int((col1.notna() & col2.notna()).sum())
            
            pairs.append({
                'Feature 1': feat1,
                'Feature 2': feat2,
                'r': round(r, 4),
                '|r|': round(abs(r), 4),
                'n_valid': n_valid,
                'Interpretazione': 'Forte' if abs(r) >= 0.80 else 'Moderata-forte'
            })

if pairs:
    df_pairs = pd.DataFrame(pairs).sort_values('|r|', ascending=False)
    print(df_pairs.to_string(index=False))
    
    # Salvataggio tabella
    output_file_table = os.path.join(output_dir, f"{dataset_name}_pearson_strong_pairs.csv")
    df_pairs.to_csv(output_file_table, index=False)
    print(f"\nTabella salvata in: {output_file_table}")
else:
    print("Nessuna coppia di feature trovata.")



TABELLA CORRELAZIONI PEARSON ((Tutte le coppie - Calcolata solo sulla componente numerica delle colonne miste)
                         Feature 1                          Feature 2       r    |r|  n_valid Interpretazione
                      NumInqLast6M              NumInqLast6Mexcl7days  0.9922 0.9922     9861           Forte
       NumTrades60Ever2DerogPubRec        NumTrades90Ever2DerogPubRec  0.8903 0.8903     9861           Forte
             NumSatisfactoryTrades                     NumTotalTrades  0.8452 0.8452     9861           Forte
             MSinceOldestTradeOpen                     AverageMInFile  0.6943 0.6943     9622  Moderata-forte
            PercentTradesNeverDelq                        MaxDelqEver  0.6392 0.6392     9861  Moderata-forte
        NumRevolvingTradesWBalance NumBank2NatlTradesWHighUtilization  0.6277 0.6277     9285  Moderata-forte
              ExternalRiskEstimate         NetFractionRevolvingBurden -0.6264 0.6264     9682  Moderata-forte
        

In [6]:
#Analasi delle relazioni tra feature e target

# Il target è categorico (Good/Bad)
target_col = 'RiskPerformance'
target_values = df[target_col]

# Prepare results list
feature_target_results = []

# Per ogni feature, calcola l'associazione appropriata con il target
for feature in all_cols:
    if feature in mixed_columns:
        # Feature MISTA - calcolare per componente numerica e categorica separatamente
        
        # ===== COMPONENTE NUMERICA (Eta: categorica(target) vs numerica(feature)) =====
        response_numeric = numeric_components[feature]
        factor_target_numeric = target_values.loc[response_numeric.index]
        factor_target_encoded = pd.factorize(factor_target_numeric)[0]
        
        if len(response_numeric) > 1:
            grand_mean = response_numeric.mean()
            ss_total = np.sum((response_numeric - grand_mean) ** 2)
            ss_between = 0
            
            for group_val in np.unique(factor_target_encoded):
                group_mask = factor_target_encoded == group_val
                group_vals = response_numeric.iloc[group_mask]
                if len(group_vals) > 0:
                    ss_between += len(group_vals) * (group_vals.mean() - grand_mean) ** 2
            
            eta_numeric = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0
        else:
            eta_numeric = np.nan
        
        feature_target_results.append({
            'feature': feature,
            'componente': 'numerica',
            'tipo_feature': 'Mista (componente numerica)',
            'metrica': 'Eta',
            'valore': eta_numeric
        })
        
        # ===== COMPONENTE CATEGORICA (Cramér's V: categorica(target) vs categorica(feature)) =====
        response_categorical = categorical_components[feature]
        factor_target_categorical = target_values.loc[response_categorical.index]
        
        if len(response_categorical) > 0:
            contingency = pd.crosstab(factor_target_categorical, response_categorical)
            chi2, p, dof, expected = chi2_contingency(contingency)
            n = contingency.sum().sum()
            min_dim = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
            cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        else:
            cramers_v = np.nan
        
        feature_target_results.append({
            'feature': feature,
            'componente': 'categorica',
            'tipo_feature': 'Mista (componente categorica)',
            'metrica': "Cramér's V",
            'valore': cramers_v
        })
    
    else:
        # Feature PURA NUMERICA - Eta (categorica vs numerica)
        response_numeric = numeric_components[feature]
        factor_target_numeric = target_values.loc[response_numeric.index]
        factor_target_encoded = pd.factorize(factor_target_numeric)[0]
        
        if len(response_numeric) > 1:
            grand_mean = response_numeric.mean()
            ss_total = np.sum((response_numeric - grand_mean) ** 2)
            ss_between = 0
            
            for group_val in np.unique(factor_target_encoded):
                group_mask = factor_target_encoded == group_val
                group_vals = response_numeric.iloc[group_mask]
                if len(group_vals) > 0:
                    ss_between += len(group_vals) * (group_vals.mean() - grand_mean) ** 2
            
            eta_numeric = np.sqrt(ss_between / ss_total) if ss_total > 0 else 0
        else:
            eta_numeric = np.nan
        
        feature_target_results.append({
            'feature': feature,
            'componente': 'numerica',
            'tipo_feature': 'Numerica (pura)',
            'metrica': 'Eta',
            'valore': eta_numeric
        })

# Crea DataFrame risultati
df_feature_target = pd.DataFrame(feature_target_results)

# Ordina per valore assoluto decrescente
df_feature_target['valore_assoluto'] = df_feature_target['valore'].abs()
df_feature_target = df_feature_target.sort_values(by='valore_assoluto', ascending=False).drop(columns=['valore_assoluto'])

# SALVATAGGIO CSV
print("\nSalvataggio CSV...")
output_file_ft = os.path.join(output_dir, f"{dataset_name}_feature_target_associazioni.csv")
df_feature_target.to_csv(output_file_ft, index=False)
print(f"  ✓ {dataset_name}_feature_target_associazioni.csv")

# CREAZIONE HEATMAP
print("\nCreazione e salvataggio heatmap...")

# Heatmap ETA (feature numeriche vs target)
df_eta_target = df_feature_target[df_feature_target['metrica'] == 'Eta'].copy()
df_eta_target = df_eta_target.set_index('feature')[['valore']].sort_values('valore', ascending=False)

plt.figure(figsize=(6, 12))
sns.heatmap(df_eta_target, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, 
            cbar_kws={'label': 'Eta Coefficient'}, linewidths=0.5)
plt.title("Eta Coefficient: Feature vs Target\n(Misura Complessiva)", fontsize=12, fontweight='bold')
plt.ylabel('Feature')
plt.xlabel('Target (RiskPerformance)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{dataset_name}_eta_target_heatmap.png"), dpi=300)
plt.close()
print(f"  ✓ {dataset_name}_eta_target_heatmap.png")

# Heatmap CRAMÉR'S V (feature categoriche miste vs target)
df_cramers_target = df_feature_target[df_feature_target['metrica'] == "Cramér's V"].copy()
df_cramers_target = df_cramers_target.set_index('feature')[['valore']].sort_values('valore', ascending=False)

plt.figure(figsize=(6, 8))
sns.heatmap(df_cramers_target, annot=True, fmt='.3f', cmap='viridis', vmin=0, vmax=1,
            cbar_kws={'label': "Cramér's V"}, linewidths=0.5)
plt.title("Cramér's V: Feature Miste vs Target\n(Componenti Categoriche)", fontsize=12, fontweight='bold')
plt.ylabel('Feature')
plt.xlabel('Target (RiskPerformance)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"{dataset_name}_cramersv_target_heatmap.png"), dpi=300)
plt.close()
print(f"  ✓ {dataset_name}_cramersv_target_heatmap.png")

print("\n" + "="*70)
print("✓ ANALISI FEATURE-TO-TARGET COMPLETATA!")
print("="*70)
print(f"Output salvato in: {output_dir}\n")



Salvataggio CSV...
  ✓ heloc_DLLM_feature_target_associazioni.csv

Creazione e salvataggio heatmap...
  ✓ heloc_DLLM_eta_target_heatmap.png
  ✓ heloc_DLLM_cramersv_target_heatmap.png

✓ ANALISI FEATURE-TO-TARGET COMPLETATA!
Output salvato in: ../../../data/processed/Fase1/AnalysisDLLM

